In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Linalg
import jax.numpy as np
import flax.linen as nn

import jax
import numpy as onp

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns

# Helper libraries
from dataclasses import dataclass
import h5py as hf

# Symbolic regression
# from pysr import PySRRegressor

## Analysis Functions

In [ ]:
temperatures = [0, 150, 273, 300, 350]

In [ ]:
@dataclass
class Measurement:
    data: np.ndarray
    uncertainty: np.ndarray

In [ ]:
def transform_coordinates(position_data: np.ndarray, reference_data: np.ndarray):
    """
    Move colloids in all frames back into a single reference frame.
    Required for RDF calculation.
    
    Parameters
    ----------
    position_data : np.ndarray (time_steps, colloids, 3)
            Colloid data to transform.
    reference_data : np.ndarray (time_steps, colloids, 3)
            Rod data acting as reference.
    """
    # Convert to polar coordinates
    def inner_fn(
        position: np.ndarray, reference
    ):
        origin = np.array([500., 500., 0.])
        distance_vector = position - origin
        distance = np.linalg.norm(distance_vector)
        normed_distance = distance_vector / distance
        
        angle = np.arctan2(
            normed_distance[0] * reference[1] - normed_distance[1] * reference[0],
            normed_distance[0] * reference[0] + normed_distance[1] * reference[1]
        )
#         angle = np.arccos(references, normed_distance)
        return np.array([distance, angle])
    
    mapped_fn = jax.vmap(inner_fn, in_axes=(0, None))
    
    outer_map_fn = jax.vmap(mapped_fn, in_axes=(0, 0))
    
    polar_coordinates = outer_map_fn(
        position_data, reference_data
    )
    
    # Convert to cartesian coordinates
    def inner_fn(
        distance: np.ndarray, angle: np.ndarray
    ):
        x = distance * np.cos(angle)
        y = distance * np.sin(angle)
        
        return np.array([x, y])
    
    mapped_fn = jax.vmap(inner_fn, in_axes=(0, 0))
    
    outer_map_fn = jax.vmap(mapped_fn, in_axes=(0, 0))
    
    cartesian_coords = outer_map_fn(
        polar_coordinates[:, :, 0], polar_coordinates[:, :, 1]
    )
        
    return cartesian_coords  

In [ ]:
def compute_rod_velocity(data: np.ndarray):
    """
    Compute rod velocity.
    
    Parameters
    ----------
    data : np.ndarray (time_steps, 1, 3)
    """
    reference = data[0]
    angles = []
    
    for i, item in enumerate(data[1:]):
        angles.append(
            np.arctan2(
            np.cross(data[i, :2], item[:2]),
            np.dot(data[i, :2], item[:2]),
        )
    )

    return np.array(angles) / 0.5

In [ ]:
def load_data(temperature: int, ensemble: int):
    """
    Load data from the hdf5 database.
    
    Parameters
    ----------
    embedding_dimension : int
            Which directory to load from.
    ensemble : int
            Which ensemble to load
    """
    root_path = f"{temperature}K/{ensemble}"
    # with hf.File(
    #     f"{root_path}/deployment/trajectory.hdf5",
    #     "r"
    # ) as db:
    #     colloid_data = db["colloids"]["Unwrapped_Positions"][:, :50, :]
    #     rod_data = db["colloids"]["Directors"][:, 55, :]

    # rewards_1 = np.load(
    #     f"{root_path}/rewards.npy", allow_pickle=True
    # )
    rewards_2 = np.load(
        f"{root_path}/rewards-long.npy", allow_pickle=True
    )
        
    return None, None, rewards_2, None

## Training Success

* Plot probability of success by vision
* Compare reward curves

In [ ]:
10000 / 200

In [ ]:
# Success probability
probabilities = {}
for temperature in temperatures:

    successes = 0
    for i in range(1, 21, 1):
        # try:
        colloid_data, rod_data, e_rewards, no_rewards = load_data(temperature, i)
        # print(e_rewards[2:].shape)  
    
        # plt.plot(np.convolve(e_rewards, np.ones(1) / 1, mode="valid"))
        plt.plot(
            np.array(np.split(e_rewards[:-1], 50)).mean(axis=1).flatten()
        )
        plt.title(f"{temperature}, {i}")
        # plt.yscale("log")
        plt.show()
        # except:
            # pass


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(
    vision_sizes, 
    [probabilities[item] for item in vision_sizes],
    "ko",
    mfc="none"
)
ax[1].errorbar(
    vision_sizes, 
    [reward_data[item].data for item in vision_sizes],
    yerr=[reward_data[item].uncertainty for item in vision_sizes],
    marker="o",
    color="k",
    mfc="none",
    ls="none",
    capsize=7,
)

# ax[1].errorbar(
#     vision_sizes, 
#     [punishment_data[item].data for item in vision_sizes],
#     yerr=[punishment_data[item].uncertainty for item in vision_sizes],
#     marker="o",
#     color="k",
#     mfc="none",
#     ls="none",
#     capsize=7,
# )

ax[0].set_xlabel("Embedding Dimension")
ax[1].set_xlabel("Embedding Dimension")
ax[0].set_ylabel("Training Probability")
ax[1].set_ylabel("Reward per episode")

plt.savefig("rewards-probability.pdf")
plt.show()

## Rod Rotation Speed

* Plot rod velocity as function of vision

In [ ]:
speeds = {}

good_models = []
for temperature in [0, 150, 273, 300, 350]:

    speed = []
    for i in range(1, 21, 1):
        _, rod_data, _, _ = load_data(temperature, i)
        velocity = (60 * 60) * (np.rad2deg(compute_rod_velocity(rod_data)) / 360)
        plt.plot(np.convolve(velocity, np.ones(300) / 300, mode="valid"))
        plt.title(f"{temperature}, {i}")
            # plt.xscale("log")
        plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2)

colours = ["#044389","#fcff4b","#ffad05","#7cafc4","#5995ed"]
linestyles = ["-", ":", "-.", "--", (5, (10, 3))]

ax = fig.add_subplot(gs[0, :])
img = sns.histplot(
    x=np.concatenate([item for item in hist_data[3].data])[:, :, 0].flatten(),
    y=np.concatenate([item for item in hist_data[3].data])[:, :, 1].flatten(),
    ax=ax,
    bins=1000,
    vmin=None,
    vmax=None,
    cbar=True,
    norm=LogNorm(),
    cmap="Spectral"

)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-100, 100)
ax.set_ylim(-75, 75)

ax = fig.add_subplot(gs[1, 0])
ax.errorbar(
    vision_sizes, 
    [60 * 60 * np.rad2deg(speeds[item].data / 2) / 360 for item in vision_sizes],
    yerr = [60 * 60 * np.rad2deg(speeds[item].uncertainty / 2) / 360 for item in vision_sizes],
    marker = "o",
    c="k",
    mfc = "none",
    ls = "none",
    capsize=7
)
ax.set_xlabel("Embedding Dimension")
ax.set_ylabel(r"Angular Velocity / rph")

ax = fig.add_subplot(gs[1, 1])
for i, hist in enumerate(hist_data):
    sns.kdeplot(
        x=np.concatenate([item for item in hist_data[hist].data])[:, :, 0].flatten(),
        label=hist,
        ls=linestyles[i],
        c=colours[i],
        ax=ax
        )
    
ax.set_xlabel("Rod Position")
ax.set_xlim(-150, 150)


fig.legend(loc=[0.87, 0.3])

plt.subplots_adjust(wspace=0.3)
plt.savefig("performance-and-strategy.png", dpi=600)
plt.show()

## Vision Interpretation

* Plot policy surfaces
* Symbolic regression on vision
    - Regression on both colloid and rod and then sum
    - Can you do regression on the sum?

In [ ]:
class ColloidEmbedding(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(features=2)(x)
    
class RodEmbedding(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(features=2)(x)
    
class ActorNet(nn.Module):
    """A simple dense model."""
    
    def setup(self):
        self.colloid_embedding = ColloidEmbedding()
        self.rod_embeding = RodEmbedding()
    
    @nn.compact
    def __call__(self, x):
        colloid_embedding = self.colloid_embedding(x[:, 0])
        rod_embedding = self.rod_embeding(x[:, 1])
        
        x = colloid_embedding + rod_embedding
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return colloid_embedding, rod_embedding, jax.nn.softmax(x)

In [ ]:
colloid_model = ActorNet()

In [ ]:
min_colloid_distance = 2.14 * 2
max_distance = 100.0
min_rod_distance = 2.14 + 100 / 59

In [ ]:
colloid_cone_values = onp.random.uniform(
    low=1 / max_distance, 
    high = 1 / min_colloid_distance, 
    size=(500, 5)
) * np.array([1., 0., 0., 0., 0.])
rod_cone_values = onp.random.uniform(
    low=1 / max_distance, 
    high = 1 / min_rod_distance, 
    size=(500, 5)
)

In [ ]:
vision_cones = np.stack((colloid_cone_values, rod_cone_values), axis=-1)

In [ ]:
all_probs = []
total_embedding = np.zeros((2,))

for ensemble in good_models:

    parameters = np.load(f"CCW/2_dimensions/{ensemble}/Models/ActorModel_0.pkl", allow_pickle=True)[0]


    colloid_embedding, rod_embedding, pre_probs = colloid_model.apply(
        {"params": parameters}, vision_cones
    )
    probs = pre_probs
    all_probs.append(pre_probs)
    total_embedding = colloid_embedding + rod_embedding
    
    fig, ax = plt.subplots(1, 4, figsize=(15, 20), subplot_kw={"projection": '3d'})

    ax[0].scatter(total_embedding[:, 0], total_embedding[:, 1], probs[:, 0])
    ax[1].scatter(total_embedding[:, 0], total_embedding[:, 1], probs[:, 1])
    ax[2].scatter(total_embedding[:, 0], total_embedding[:, 1], probs[:, 2])
    ax[3].scatter(total_embedding[:, 0], total_embedding[:, 1], probs[:, 3])


    ax[0].set_title("Rotate CW")
    ax[1].set_title("Translate")
    ax[2].set_title("Rotate CCW")
    ax[3].set_title("Do Nothing")

    plt.show()


In [ ]:
x = ["CW", "Translate", "CCW", "Do Nothing"]
y = onp.sum(all_probs, axis=0).sum(axis=0) / (13 * 5000)

plt.bar(x, y)